## Seccomp profiles

**seccomp** (secure computing mode) is a kernel feature that filters *which syscalls* a process may make — a layer below capabilities. Where a capability gates a *privilege*, seccomp gates the individual *system calls* themselves. Docker applies its **`default.json`** profile to every container automatically, blocking ~50 syscalls with a history of CVEs or no place in a normal app (`mount`, `umount2`, `keyctl`, `reboot`, …).

For most containers **the default is enough** — you rarely touch seccomp. To tighten further, point at a custom allow-list profile:

```bash
docker run --security-opt seccomp=./my-profile.json myorg/api
```

A custom profile lets through only the syscalls your app actually uses; tools like `falco` or `oci-seccomp-bpf-hook` can trace the app under normal load and emit a minimal profile automatically. This is the tightest syscall filtering you can apply, but it's high-effort — reserve it for genuinely sensitive workloads.

The **anti-pattern to resist**: disabling seccomp entirely.

```bash
docker run --security-opt seccomp=unconfined myorg/api   # DON'T
```

This is the "it didn't work so I turned it off" fix you'll see all over Stack Overflow. It removes an entire defensive layer to paper over one blocked syscall. The right move is the opposite — find *which* syscall is blocked and either fix the app or add that one call to the profile. Seccomp is on by default for a reason; keep it on, and tune rather than remove.